# Phase 2: Data Ingestion & Baseline Detectors

**Project:** Cross-Modal Kinematic–Electromagnetic Coupling for GNSS Spoofing Detection (CKEC-GNSS)  
**Notebook Reference:** `01_baseline_detectors.ipynb`  
**Deliverable:** Phase 2 Reproducible Baseline Notebook  

---

## 1. Overview and Problem Statement

Global Navigation Satellite System (GNSS) spoofing attacks inject counterfeit radio-frequency signals designed to manipulate a target receiver's Position, Velocity, and Time (PVT) solution.

Conventional anti-spoofing techniques often rely on:
1. **Multi-antenna spatial arrays** (Angle-of-Arrival) — *unavailable* in mass-market single-antenna receivers (as verified in our Phase 1 audit of `FGI-SpoofRepo`, `Jammertest-2024`, and `Yunnan-2024`).
2. **Pure power / AGC thresholding** — vulnerable to power-matched or gradual power-ramped spoofers.
3. **Pure carrier phase / Doppler residuals** — prone to false alarms during legitimate platform dynamics or multipath.
4. **Heuristic quotient statistics** ($|\Delta \phi_t| / (|\Delta \text{AGC}_t| + \epsilon)$) — inherently unstable when $\Delta \text{AGC}_t \approx 0$.

### Notebook Objectives:
1. Generate controlled GNSS observation streams (carrier-phase and front-end gain control).
2. Apply rigorous signal preprocessing: phase unwrapping, AGC smoothing, and temporal differencing.
3. Perform **pre-attack calibration** strictly on certified authentic intervals to establish $\mu_{\text{cal}}$ and $\sigma_{\text{cal}}$.
4. Implement and evaluate baseline detectors:
   - **D1 — Raw Ratio Baseline:** $J_t = |\Delta \phi_t| / (|\Delta \text{AGC}_t| + \epsilon)$
   - **D2 — Phase-Only Detector:** $|Z_{\phi, t}|$
   - **D3a — AGC-Only Detector:** $|Z_{\text{agc}, t}|$
5. Demonstrate false-alarm calibration targeting $\text{FAR} = 0.01$ (1%) on benign validation data.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ckec.simulation import generate_abrupt_spoofing, generate_benign_multipath
from ckec.preprocessing import unwrap_phase, smooth_agc, estimate_benign_stats, handle_missing
from ckec.features import delta_phase, delta_agc, normalised_phase, normalised_agc, raw_ratio, build_feature_matrix
from ckec.detectors import RawRatioDetector, PhaseOnlyDetector, AGCOnlyDetector
from ckec.evaluation import compute_metrics, bootstrap_metrics

# Matplotlib configuration for publication-ready plots
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "figure.figsize": (11, 5),
})

print("CKEC modules and dependencies imported successfully.")


## 2. Controlled Scenario Simulation

We simulate two scenarios:
1. **Abrupt Spoofing:** Authentic signal for 1,500 samples, followed by a spoofing takeover introducing an abrupt phase offset ($\Delta \phi = 1.5$ rad) and gain step ($\Delta A = 3.0$ dB).
2. **Benign Multipath:** Authentic signal undergoing realistic environmental multipath fluctuations without spoofing.


In [ ]:
n_samples = 3000
onset_sample = 1500
seed = 42

df_spoof, meta_spoof = generate_abrupt_spoofing(
    n_samples=n_samples,
    onset=onset_sample,
    noise_std_phase=0.05,
    noise_std_agc=0.2,
    spoof_phase_offset=1.5,
    spoof_agc_jump_db=3.0,
    seed=seed,
)

df_mp, meta_mp = generate_benign_multipath(
    n_samples=n_samples,
    mp_amplitude=0.3,
    seed=seed,
)

print("Generated scenarios:")
print(f" - Abrupt Spoofing: {len(df_spoof)} samples (onset at t={df_spoof['time_s'].iloc[onset_sample]:.1f}s)")
print(f" - Benign Multipath: {len(df_mp)} samples (all benign)")
df_spoof.head()


## 3. Preprocessing & Benign Calibration

### Key Rules:
- Thresholds and normalization statistics **must be estimated on benign calibration intervals**, never contaminated by attack data.
- Here, we designate the first 500 samples (50 seconds) as the certified authentic calibration window.


In [ ]:
calib_samples = 500

# 1. Carrier phase unwrapping
phase_unwrapped = unwrap_phase(df_spoof["carrier_phase"])

# 2. AGC smoothing (rolling window of 5 samples)
agc_smoothed = smooth_agc(df_spoof["agc"], window=5)

# 3. Temporal differences
d_phi = delta_phase(phase_unwrapped)
d_agc = delta_agc(agc_smoothed)

# 4. Calibration statistics from authentic interval
mu_phi, sigma_phi = estimate_benign_stats(d_phi, calib_samples)
mu_agc, sigma_agc = estimate_benign_stats(d_agc, calib_samples)

# 5. Normalised innovation z-scores
z_phi = normalised_phase(d_phi, mu_phi, sigma_phi)
z_agc = normalised_agc(d_agc, mu_agc, sigma_agc)
r_raw = raw_ratio(d_phi, d_agc, epsilon=1e-4)

# Build unified feature matrix
feat_df = build_feature_matrix(z_phi=z_phi, z_agc=z_agc)
feat_df["delta_phi"] = d_phi
feat_df["delta_agc"] = d_agc
feat_df["raw_ratio"] = r_raw
feat_df["attack_label"] = df_spoof["attack_label"].values
feat_df["time_s"] = df_spoof["time_s"].values
feat_df = feat_df.dropna()

print(f"Calibration Phase:  mean={mu_phi:.5f}, std={sigma_phi:.5f}")
print(f"Calibration AGC:    mean={mu_agc:.5f}, std={sigma_agc:.5f}")


## 4. Visualizing Signal Innovations and Raw Ratio Instability

Notice how:
- The raw quotient $J_t = |\Delta \phi_t| / (|\Delta \text{AGC}_t| + \epsilon)$ suffers from erratic, spurious spikes whenever $|\Delta \text{AGC}_t| \approx 0$ during completely benign tracking.
- Normalized innovations $Z_\phi$ and $Z_A$ remain tightly bounded in the benign zone ($|Z| < 3$).


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

t = feat_df["time_s"]
onset_t = df_spoof["time_s"].iloc[onset_sample]

axs[0].plot(t, feat_df["z_phi"], label=r"$Z_\phi$ (Phase Innovation)", color="#1f77b4")
axs[0].axvline(onset_t, color="red", linestyle="--", label="Spoofing Onset")
axs[0].set_ylabel("Z-score")
axs[0].set_title("Normalized Carrier-Phase Innovations")
axs[0].legend(loc="upper left")

axs[1].plot(t, feat_df["z_agc"], label=r"$Z_{\rm agc}$ (AGC Innovation)", color="#ff7f0e")
axs[1].axvline(onset_t, color="red", linestyle="--")
axs[1].set_ylabel("Z-score")
axs[1].set_title("Normalized AGC Power Innovations")
axs[1].legend(loc="upper left")

axs[2].plot(t, feat_df["raw_ratio"], label=r"D1 Raw Ratio $|\Delta \phi| / (|\Delta A| + \epsilon)$", color="#d62728")
axs[2].axvline(onset_t, color="red", linestyle="--")
axs[2].set_ylabel("Ratio Value")
axs[2].set_xlabel("Time (seconds)")
axs[2].set_title("D1 Raw Ratio — Demonstrating Erratic Spikes Under Small Denominators")
axs[2].legend(loc="upper left")

plt.tight_layout()
plt.show()


## 5. Detector Fitting & Strict False-Alarm Calibration

We partition the data into:
- **Validation Split:** Samples 0–1,200 (strictly benign pre-attack) for threshold calibration targeting $\text{FAR} = 0.01$ (1%).
- **Test Split:** Samples 1,200–3,000 (spanning the benign-to-spoofing transition).


In [ ]:
val_data = feat_df.iloc[:1200]
test_data = feat_df.iloc[1200:]

y_val = val_data["attack_label"].to_numpy()
y_test = test_data["attack_label"].to_numpy()

d1 = RawRatioDetector(epsilon=1e-4)
d2 = PhaseOnlyDetector()
d3 = AGCOnlyDetector()

target_far = 0.01

d1.fit(val_data, y_val, target_far=target_far)
d2.fit(val_data, y_val, target_far=target_far)
d3.fit(val_data, y_val, target_far=target_far)

print(f"Calibrated Thresholds (FAR = {target_far*100:.1f}%):")
print(f" - D1 Raw Ratio:  {d1._threshold:.4f}")
print(f" - D2 Phase Only: {d2._threshold:.4f}")
print(f" - D3a AGC Only:  {d3._threshold:.4f}")


## 6. Held-Out Evaluation on Test Partition

We evaluate detection probability ($P_D$), false-alarm rate ($FAR$), precision, recall, and $F_1$ score.


In [ ]:
detectors = {"D1 Raw Ratio": d1, "D2 Phase Only": d2, "D3a AGC Only": d3}
results = []

for name, det in detectors.items():
    preds = det.predict(test_data).to_numpy()
    scores = det.score(test_data).to_numpy()
    
    m = compute_metrics(y_test, preds, scores, detector_name=name, split="test", target_far=target_far)
    ci = bootstrap_metrics(y_test, preds, n_iter=500, seed=42)
    
    results.append({
        "Detector": name,
        "Detection Prob (P_D)": f"{m.detection_probability:.3f}",
        "False Alarm Rate (FAR)": f"{m.false_alarm_rate:.3f}",
        "Precision": f"{m.precision:.3f}",
        "F1 Score": f"{m.f1:.3f}",
        "ROC-AUC": f"{m.roc_auc:.3f}",
        "PR-AUC": f"{m.pr_auc:.3f}",
        "FAR 95% CI": f"[{ci['false_alarm_rate'][0]:.3f}, {ci['false_alarm_rate'][1]:.3f}]",
    })

res_df = pd.DataFrame(results)
res_df


## 7. Conclusions from Baseline Analysis

1. **Strict Calibration Adherence:** Calibrating thresholds on benign validation data ensures that the empirical test FAR aligns closely with the target 1% margin.
2. **D1 Instability:** The raw ratio quotient exhibits extreme volatility when $\Delta A \approx 0$, leading to unreliable thresholding.
3. **Single-Modality Limitations:** Neither phase-only ($D2$) nor AGC-only ($D3a$) alone provides optimal detection against complex spoofing signatures.
4. **Next Step (Phase 3):** In `02_ckec_comparative_evidence.ipynb`, we evaluate the candidate cross-modal coupling mechanism ($D4$ and $D6$), demonstrating how temporal correlation collapse provides superior spoofing discrimination.
